# VascX + EyeLiner — 다중 환자 종단 registration

**목표**: 같은 환자의 여러 방문 이미지를 첫 방문 기준으로 정렬해서 vessel/AV/disc mask 를 통일된 좌표계로 뽑는다.

**파이프라인**:
1. 임상 CSV → 방문 수 top-2 환자 선정
2. base preprocess (모든 이미지, **512 RGB+CE**)
3. VascX 추론 (fovea, discedge, quality, disc/av/vessels mask — 모두 512 grid)
4. 인벤토리: (환자, 좌우안) 로 그룹화, 시간순 정렬
5. 각 (환자, 좌우안) 에 대해 첫 방문 = fixed, 나머지 = moving → EyeLiner 정렬
6. **`theta` 만 CSV 저장** + aligned mask 저장 (aligned RGB 는 저장 안 함 → warp on-demand)

**디스크 절약**:
- 1024 → 512: 이미지당 ~4× 작아짐 (models 은 어차피 512 forward)
- aligned RGB 저장 안 함: `theta` + base RGB 만 있으면 언제든 재현 가능
- aligned mask 만 저장 (feature 추출용, PNG 라 압축 잘 됨)

**좌우안 판정**: `discedge.x < fovea.x` → Right eye (OD),  
`discedge.x > fovea.x` → Left eye (OS)

## 0. Imports + 경로


In [ ]:
import importlib
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

from rtnls_fundusprep.preprocessor import parallel_preprocess
from rtnls_inference import (
    ClassificationEnsemble,
    HeatmapRegressionEnsemble,
    RegressionEnsemble,
    SegmentationEnsemble,
)
from rtnls_inference.utils import decollate_batch


In [ ]:
# ── 전역 설정 ──
SQUARE_SIZE = 512     # base preprocess + 모델 mask/좌표 grid (1024 → 512 로 낮춤)

# ── 소스 (읽기 전용) ──
src_path  = Path("g:/RAG/20230118_HCS_Fundus_image")
csv_path  = Path("G:/RAG/longitudinal/csv/longitudinal_final_260820_reclassified.csv")

# ── 전처리 결과 (RGB + CE) — 512 grid ──
prep_root = Path(f"g:/RAG/project_1/output/preprocessed_{SQUARE_SIZE}") / src_path.name
rgb_path  = prep_root / "rgb"
ce_path   = prep_root / "ce"

# ── base 추론 산출물 (registration 전) ──
out_root      = Path(f"g:/RAG/project_1/output/vascx_output_{SQUARE_SIZE}") / src_path.name
av_path       = out_root / "av"
vessels_path  = out_root / "vessels"
discs_path    = out_root / "discs"

# ── 정렬 후 산출물 (registration 후) — mask 만 저장, RGB 는 저장 안 함 ──
aligned_root  = Path(f"g:/RAG/project_1/output/aligned_{SQUARE_SIZE}") / src_path.name

for p in [rgb_path, ce_path, av_path, vessels_path, discs_path, aligned_root]:
    p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device, "| square_size:", SQUARE_SIZE)
print("prep:   ", prep_root)
print("out:    ", out_root)
print("aligned:", aligned_root)

## 1. 임상 CSV 로드 + 환자 2 명 선정

방문 수가 많은 상위 2 명 (테스트 풍부성).


In [ ]:
data = pd.read_csv(csv_path, low_memory=False)

counts = data.groupby("ID").size().sort_values(ascending=False)
TOP_N = 3
patient_ids = counts.head(TOP_N).index.tolist()
print(f"총 환자 수: {data['ID'].nunique():,}, 이미지 수: {len(data):,}")
print(f"선정 {TOP_N} 명 (방문 수 상위):")
for pid in patient_ids:
    print(f"  ID={pid}: {counts[pid]} 장")


In [ ]:
# 선정 환자의 전체 이미지 rows
cohort = data[data["ID"].isin(patient_ids)].copy()
cohort["LAB_DTM"] = pd.to_datetime(cohort["LAB_DTM"])
cohort = cohort.sort_values(["ID", "LAB_DTM", "shot"]).reset_index(drop=True)
cohort[["ID", "Filename", "LAB_DTM", "shot", "Age"]].head(20)


In [ ]:
cohort

## 2. 전처리 (base) — 원본 → `SQUARE_SIZE` RGB + CE

이미 처리된 파일은 스킵.

In [ ]:
files = [src_path / fn for fn in cohort["Filename"]]
existing = {p.stem for p in rgb_path.glob("*.png")}
todo = [f for f in files if f.stem not in existing]
print(f"total: {len(files)}, cached: {len(files)-len(todo)}, todo: {len(todo)}")

if todo:
    parallel_preprocess(
        todo, rgb_path=rgb_path, ce_path=ce_path,
        square_size=SQUARE_SIZE, n_jobs=4,
    )
    print(f"preprocessed: {len(todo)} @ {SQUARE_SIZE}x{SQUARE_SIZE}")
else:
    print("전부 캐시됨")

In [ ]:
# 이번 세션 파일들만 추론 대상
ids = [f.stem for f in files]
rgb_paths = [rgb_path / f"{i}.png" for i in ids]
ce_paths  = [ce_path  / f"{i}.png" for i in ids]
missing = [p for p in rgb_paths + ce_paths if not p.exists()]
assert not missing, f"전처리 실패: {missing[:3]}"
paired_paths = list(zip(rgb_paths, ce_paths))
print(f"이미지 {len(paired_paths)} 쌍 준비")


## 3. VascX 모델 로드 (7 개)


In [ ]:
weights_root = Path("../weights/vascx")

ensemble_quality  = ClassificationEnsemble.from_release(str(weights_root/"quality.pt")).to(device).eval()
ensemble_av       = SegmentationEnsemble.from_release(str(weights_root/"av_july24.pt")).to(device).eval()
ensemble_vessels  = SegmentationEnsemble.from_release(str(weights_root/"vessels_july24.pt")).to(device).eval()
ensemble_disc     = SegmentationEnsemble.from_release(str(weights_root/"disc_july24.pt")).to(device).eval()
ensemble_fovea    = HeatmapRegressionEnsemble.from_release(str(weights_root/"fovea_july24.pt")).to(device).eval()
ensemble_discedge = HeatmapRegressionEnsemble.from_release(str(weights_root/"discedge_july24.pt")).to(device).eval()
ensemble_odfd     = RegressionEnsemble.from_release(str(weights_root/"odfd_march25.pt")).to(device).eval()

# ── config 의 square_size 를 SQUARE_SIZE 로 override ──
# 기본 config 에는 square_size=1024 로 저장돼 있어 mask/keypoint 가 1024 로 undo_resize 됨.
# 우리는 입력을 SQUARE_SIZE 로 preprocess 했으니 undo 도 같은 grid 로 맞춰야 함.
for ens in [ensemble_av, ensemble_vessels, ensemble_disc, ensemble_fovea, ensemble_discedge, ensemble_odfd]:
    tt = ens.config["datamodule"].setdefault("test_transform", {})
    tt["square_size"] = SQUARE_SIZE
    tt.setdefault("resize", SQUARE_SIZE)   # forward 해상도 (모델은 원래 512 로 학습)

print(f"loaded 7 models (square_size overridden → {SQUARE_SIZE})")

## 4. 추론 — 좌우안 판정을 위해 fovea + discedge 는 필수

마스크(av/vessels/discs) 는 registration 전에 뽑아둠 (뒤에서 warp).


In [ ]:
# 4-1. Fovea (RGB+CE)
df_fovea = ensemble_fovea.predict_preprocessed(
    paired_paths, ids=ids, num_workers=2, batch_size=8,
)
df_fovea.columns = ["x_fovea", "y_fovea"]

# 4-2. Discedge (RGB+CE)
df_discedge = ensemble_discedge.predict_preprocessed(
    paired_paths, ids=ids, num_workers=2, batch_size=8,
)
df_discedge.columns = ["x_discedge", "y_discedge"]

# 4-3. ODFD (RGB only)
df_odfd = ensemble_odfd.predict_preprocessed(
    [str(p) for p in rgb_paths], ids=ids, num_workers=2, batch_size=8,
)
df_odfd.columns = ["v_odfd"]

# 4-4. Quality (RGB only) — 참고용, filter 안 씀
dl_q = ensemble_quality._make_inference_dataloader(
    [str(p) for p in rgb_paths], ids=ids, num_workers=2, preprocess=False, batch_size=16,
)
qids, qrows = [], []
with torch.no_grad():
    for batch in tqdm(dl_q, desc="quality"):
        if len(batch) == 0: continue
        im = batch["image"].to(device)
        q = ensemble_quality.predict_step(im).mean(dim=0)
        for it in decollate_batch({"id": batch["id"], "quality": q}):
            qids.append(it["id"]); qrows.append(it["quality"].tolist())
df_quality = pd.DataFrame(qrows, index=qids, columns=["q1", "q2", "q3"])
print("scalar/keypoint 추론 완료")


In [ ]:
# 4-5. Segmentation masks (av, vessels, disc)
ensemble_av.predict_preprocessed(paired_paths, ids=ids, dest_path=av_path, num_workers=2, batch_size=8)
ensemble_vessels.predict_preprocessed(paired_paths, ids=ids, dest_path=vessels_path, num_workers=2, batch_size=8)
ensemble_disc.predict_preprocessed(paired_paths, ids=ids, dest_path=discs_path, num_workers=2, batch_size=8)
print("masks:",
      "av", len(list(av_path.glob('*.png'))),
      "vessels", len(list(vessels_path.glob('*.png'))),
      "disc", len(list(discs_path.glob('*.png'))))


## 5. 인벤토리 구축

각 이미지에 대해:
- **Disc 중심** = disc segmentation mask 의 centroid (여러 blob 있으면 최대 것)
  - 이유: `discedge` (single keypoint) 는 unstable, mask centroid 는 수십~수백 픽셀 평균 → robust
  - Fallback: mask 가 비어있거나 너무 작으면 (`<50 px`) discedge keypoint 사용
  - Bonus: disc **diameter** 도 같이 뽑음 (나중에 DD 단위 정규화용)
- **좌우안 판정**: `sep_x = disc.x - fovea.x`.  
  `sep_x < 0` → **R (right eye, OD)**, `sep_x > 0` → **L (left eye, OS)**, `|sep_x| < min_frac × W` → **unknown**
- `(patient_id, eye)` 로 그룹화 후 `LAB_DTM` 순 정렬

In [ ]:
from scipy import ndimage as ndi

MIN_SEP_FRAC = 0.02        # |disc.x - fovea.x| 가 이보다 작으면 laterality 불확실
MIN_DISC_PX  = 50          # disc mask 픽셀이 이보다 적으면 fallback

def disc_geometry(disc_mask, min_area_px=MIN_DISC_PX):
    """Disc mask → centroid + area-equivalent diameter.
    가장 큰 blob 만 유지 (bright artifact 제거).
    found=False → 사용 불가 (비었거나 너무 작음)."""
    m = disc_mask.astype(bool)
    if m.sum() < min_area_px:
        return dict(cx=np.nan, cy=np.nan, diameter=np.nan,
                    area_px=float(m.sum()), found=False)
    lab, n = ndi.label(m)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (int(np.argmax(sizes)) + 1)
    area = float(m.sum())
    cy, cx = ndi.center_of_mass(m)
    return dict(cx=float(cx), cy=float(cy),
                diameter=2.0 * np.sqrt(area / np.pi),
                area_px=area, found=True)

def determine_laterality(x_fovea, x_disc, W=SQUARE_SIZE, min_frac=MIN_SEP_FRAC):
    sep = x_disc - x_fovea
    if abs(sep) < min_frac * W:
        return "unknown", sep
    return ("L" if sep > 0 else "R"), sep

# 이미지 metadata 조립
rows = []
for _, r in cohort.iterrows():
    stem = Path(r["Filename"]).stem
    if stem not in df_fovea.index or stem not in df_discedge.index:
        continue
    fx, fy = df_fovea.loc[stem, ["x_fovea", "y_fovea"]]

    # ── disc 중심: mask centroid 우선, 실패 시 discedge keypoint fallback ──
    disc_mask_path = discs_path / f"{stem}.png"
    if disc_mask_path.exists():
        dg = disc_geometry(np.array(Image.open(disc_mask_path)))
    else:
        dg = dict(cx=np.nan, cy=np.nan, diameter=np.nan, area_px=0, found=False)

    if dg["found"]:
        dx, dy = dg["cx"], dg["cy"]
        disc_source = "mask"
    else:
        dx = float(df_discedge.loc[stem, "x_discedge"])
        dy = float(df_discedge.loc[stem, "y_discedge"])
        disc_source = "keypoint_fallback"

    eye, sep = determine_laterality(fx, dx)
    q = df_quality.loc[stem].tolist() if stem in df_quality.index else [np.nan]*3
    rows.append({
        "id": stem, "patient": r["ID"], "date": r["LAB_DTM"],
        "shot": r["shot"], "filename": r["Filename"],
        "eye": eye, "sep_x": sep,
        "x_fovea": fx, "y_fovea": fy,
        "x_disc": dx, "y_disc": dy,
        "disc_diameter": dg["diameter"], "disc_area_px": dg["area_px"],
        "disc_source": disc_source,
        # 참고용: 원본 discedge keypoint 도 같이 보관
        "x_discedge_kp": float(df_discedge.loc[stem, "x_discedge"]),
        "y_discedge_kp": float(df_discedge.loc[stem, "y_discedge"]),
        "v_odfd": df_odfd.loc[stem, "v_odfd"] if stem in df_odfd.index else np.nan,
        "q1": q[0], "q2": q[1], "q3": q[2],
    })

inv_df = pd.DataFrame(rows).sort_values(["patient", "eye", "date", "shot"]).reset_index(drop=True)

n_fallback = (inv_df["disc_source"] == "keypoint_fallback").sum()
print(f"disc centroid: mask={len(inv_df)-n_fallback}, keypoint fallback={n_fallback}")
print(f"disc_diameter (px): median={inv_df['disc_diameter'].median():.1f}, "
      f"min={inv_df['disc_diameter'].min():.1f}, max={inv_df['disc_diameter'].max():.1f}")
inv_df
inv_df.to_csv(aligned_root / "inventory.csv", index=False)


In [ ]:
# 인벤토리를 dict 구조로도
inventory = {}
for (pid, eye), grp in inv_df.groupby(["patient", "eye"], sort=False):
    inventory.setdefault(pid, {})[eye] = grp.to_dict("records")

for pid in patient_ids:
    print(f"\n환자 {pid}:")
    for eye, visits in inventory.get(pid, {}).items():
        print(f"  {eye}: {len(visits)} 장", [v["date"].strftime("%Y-%m-%d") for v in visits])


## 6. EyeLiner 준비 (local `../EyeLiner/` 사용)

`fundus-vessel-features-2/EyeLiner/` 아래 eyeliner + lightglue 를 번들해뒀음.  
Original repo (`g:/RAG/longitudinal/EyeLiner`) 참조 필요 없음.


In [ ]:
# ── Local EyeLiner (fundus-vessel-features-2/EyeLiner/) 로드 ──
EYELINER_ROOT = Path("../EyeLiner").resolve()
for p in [EYELINER_ROOT]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))

# eyeliner/detectors.py 가 `from .lightglue import ...` 로 하지만
# 우리는 lightglue 를 자매 폴더로 분리해뒀음 → sys.modules 로 우회
if "eyeliner.lightglue" not in sys.modules:
    sys.modules["eyeliner.lightglue"]       = importlib.import_module("lightglue")
    sys.modules["eyeliner.lightglue.utils"] = importlib.import_module("lightglue.utils")

from eyeliner import EyeLinerP

EL_SIZE = 256
eyeliner_api = EyeLinerP(
    kp_method="splg", reg="affine",
    image_size=(3, EL_SIZE, EL_SIZE), device=device,
)
print("EyeLinerP ready (local EyeLiner)")


## 7. 환자별 registration

각 (환자, 좌우안) 그룹에서 **첫 방문 = fixed**, 나머지 = moving 을 정렬.

**저장 정책** (용량 절약):
- **aligned RGB 는 저장 안 함** — `theta` + base RGB 만 있으면 재현 가능
- **aligned mask (av/vessels/discs)** 는 저장 → downstream feature 추출용
- `theta` (3×3 matrix) 는 `registration_log.csv` 에 flat 하게 기록

In [ ]:
def prep_tensor(img_np, size=EL_SIZE):
    a = cv2.resize(img_np.astype(np.float32) / 255.0, (size, size))
    return torch.tensor(a).permute(2, 0, 1).unsqueeze(0).to(device)

def rescale_theta(theta_256, H, W, size=EL_SIZE):
    S = np.array([[W/size,0,0],[0,H/size,0],[0,0,1]], dtype=np.float64)
    return S @ theta_256 @ np.linalg.inv(S)

def warp_image(img, theta, H, W, is_mask=False):
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    return cv2.warpAffine(img, theta[:2, :], (W, H),
                          flags=flags, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def register_pair(rgb_fixed, rgb_moving):
    H, W = rgb_fixed.shape[:2]
    tf, tm = prep_tensor(rgb_fixed), prep_tensor(rgb_moving)
    theta, cache = eyeliner_api({"fixed_input": tf, "moving_input": tm})
    theta = theta.squeeze().cpu().numpy()
    theta_full = rescale_theta(theta, H, W)
    n_kp = cache["kp_fixed"].shape[1]
    return theta_full, n_kp


In [ ]:
def save_mask_png(mask, path):
    """label mask → PNG optimize=True (18% 추가 절약)."""
    Image.fromarray(mask).save(path, format="PNG", optimize=True)

def register_group(visits, patient, eye, save_root):
    """visits: LAB_DTM 순 정렬된 dict 리스트. 첫 방문 = fixed.
    aligned RGB 는 저장 안 함, aligned mask 만 저장 (feature 추출용)."""
    if len(visits) < 2:
        print(f"  [{patient}/{eye}] {len(visits)} 장 뿐, 스킵")
        return []

    out_dir = save_root / str(patient) / eye
    for tag in ["av", "vessels", "discs"]:
        (out_dir / tag).mkdir(parents=True, exist_ok=True)

    fixed = visits[0]
    fixed_rgb = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    H, W = fixed_rgb.shape[:2]

    # fixed 는 identity — mask 만 그대로 복사 (aligned mask 폴더에)
    for tag, mask_dir in [("av", av_path), ("vessels", vessels_path), ("discs", discs_path)]:
        m = np.array(Image.open(mask_dir / f"{fixed['id']}.png"))
        save_mask_png(m, out_dir / tag / f"{fixed['id']}.png")

    records = [{**fixed, "is_fixed": True, "n_kp": np.nan,
                "theta": np.eye(3).flatten().tolist()}]

    for v in visits[1:]:
        moving_rgb = np.array(Image.open(rgb_path / f"{v['id']}.png"))
        try:
            theta, n_kp = register_pair(fixed_rgb, moving_rgb)
        except Exception as e:
            print(f"  [{patient}/{eye}] {v['id']} registration 실패: {e}")
            continue

        # aligned mask 만 저장 (nearest-neighbor 로 라벨 보존)
        for tag, mask_dir in [("av", av_path), ("vessels", vessels_path), ("discs", discs_path)]:
            m = np.array(Image.open(mask_dir / f"{v['id']}.png"))
            wm = warp_image(m, theta, H, W, is_mask=True)
            save_mask_png(wm, out_dir / tag / f"{v['id']}.png")

        records.append({**v, "is_fixed": False, "n_kp": n_kp,
                        "theta": theta.flatten().tolist()})
    return records

# 실행
all_records = []
for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye == "unknown":
            print(f"[{pid}/unknown] laterality 판정 실패, 스킵")
            continue
        print(f"[{pid}/{eye}] fixed={visits[0]['id']}, moving={len(visits)-1} 장")
        recs = register_group(visits, pid, eye, aligned_root)
        all_records.extend(recs)

# theta 를 t00..t22 컬럼으로 펼쳐서 저장
reg_df = pd.DataFrame(all_records)
if len(reg_df):
    theta_cols = [f"t{i}{j}" for i in range(3) for j in range(3)]
    reg_df[theta_cols] = pd.DataFrame(reg_df["theta"].tolist(), index=reg_df.index)
    reg_df = reg_df.drop(columns=["theta"])
reg_df.to_csv(aligned_root / "registration_log.csv", index=False)
print(f"\n총 {len(reg_df)} 장 처리, log 저장: {aligned_root/'registration_log.csv'}")

## 8. 환자별 timeline 시각화

각 (환자, 좌우안) 의 원본 / 정렬된 warp / overlay 를 한 줄로.  
aligned RGB 는 저장 안 했으므로 `registration_log.csv` 의 `theta` 로 **on-demand warp**.

In [ ]:
# theta 조회 helper
theta_cols = [f"t{i}{j}" for i in range(3) for j in range(3)]
def get_theta(image_id):
    row = reg_df[reg_df["id"] == image_id]
    if len(row) == 0: return None
    return row.iloc[0][theta_cols].values.astype(np.float64).reshape(3, 3)

def make_overlay(fixed_img, moving_img):
    fg = cv2.cvtColor(fixed_img, cv2.COLOR_RGB2GRAY)
    mg = cv2.cvtColor(moving_img, cv2.COLOR_RGB2GRAY)
    out = np.zeros((*fg.shape, 3), dtype=np.uint8)
    out[..., 0] = fg; out[..., 1] = mg; out[..., 2] = mg
    return out

def plot_patient_eye(patient, eye, visits):
    if len(visits) < 2: return
    fixed = visits[0]
    fixed_rgb = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    H, W = fixed_rgb.shape[:2]
    n = len(visits)

    fig, axes = plt.subplots(3, n, figsize=(3.5*n, 10))
    if n == 1: axes = axes[:, None]
    for i, v in enumerate(visits):
        orig = np.array(Image.open(rgb_path / f"{v['id']}.png"))
        axes[0, i].imshow(orig)
        axes[0, i].set_title(f"{v['date'].strftime('%Y-%m-%d')}\noriginal")

        if i == 0:
            aligned = fixed_rgb
            axes[1, i].set_title("FIXED")
        else:
            theta = get_theta(v["id"])
            if theta is None:
                for r in [1, 2]:
                    axes[r, i].axis("off"); axes[r, i].set_title("(skipped)")
                continue
            aligned = warp_image(orig, theta, H, W)
            axes[1, i].set_title("aligned (warp on-demand)")

        axes[1, i].imshow(aligned)
        axes[2, i].imshow(make_overlay(fixed_rgb, aligned))
        axes[2, i].set_title("overlay vs fixed")

    for ax in axes.flat: ax.axis("off")
    fig.suptitle(f"Patient {patient} — {eye} eye ({n} visits)", fontsize=14)
    plt.tight_layout(); plt.show()

for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye != "unknown":
            plot_patient_eye(pid, eye, visits)

## 9. 정렬된 mask overlay 확인

같은 (환자, 좌우안) 의 AV mask 를 시간에 따라 겹쳐서 표시. 잘 정렬되면 vessel tree 가 비슷한 위치에.


In [ ]:
def plot_mask_timeline(patient, eye, visits, mask_tag="av"):
    if len(visits) < 2: return
    mdir = aligned_root / str(patient) / eye / mask_tag
    fig, axes = plt.subplots(1, len(visits), figsize=(3.5*len(visits), 4))
    if len(visits) == 1: axes = [axes]
    for i, v in enumerate(visits):
        p = mdir / f"{v['id']}.png"
        if not p.exists():
            axes[i].axis("off"); continue
        mask = np.array(Image.open(p))
        if mask_tag == "av":
            # 0=bg 1=A(red) 2=V(blue) 3=cross(magenta)
            rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
            rgb[mask == 1] = [255, 30, 30]
            rgb[mask == 2] = [30, 70, 255]
            rgb[mask == 3] = [200, 30, 200]
            axes[i].imshow(rgb)
        else:
            axes[i].imshow(mask, cmap="gray")
        axes[i].set_title(v['date'].strftime('%Y-%m-%d'))
        axes[i].axis("off")
    fig.suptitle(f"Patient {patient} {eye} — {mask_tag} (aligned)", fontsize=13)
    plt.tight_layout(); plt.show()

for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye != "unknown":
            plot_mask_timeline(pid, eye, visits, "av")


## 9-QC. Alignment 정량 진단

**진단 1: Fixed vs 각 visit vessel 오버레이**
- **빨강** = fixed 시점의 vessel mask
- **초록** = 정렬된 다른 visit 의 vessel mask
- **노랑** (빨강+초록) = 겹침
- Alignment 완벽하면 대부분 노랑, 나쁘면 빨강/초록이 분리됨

**진단 2: Disc/Fovea centroid 안정성**
- 각 visit 의 aligned vessel mask 에서 disc mask centroid 계산
- Alignment 잘 되면 centroid 위치가 fixed 대비 몇 픽셀 이내로 일정


In [ ]:
# 진단 1: fixed vs 각 visit 의 aligned vessel 오버레이
# 빨강=fixed, 초록=aligned moving, 노랑=overlap

for (pid, eye), roi in patient_roi.items():
    visits = inventory[pid][eye]
    if len(visits) < 2: continue
    fixed = visits[0]
    fixed_vessel = np.array(Image.open(aligned_root / str(pid) / eye / "vessels" / f"{fixed['id']}.png")) > 0

    # 몇 visit 만 샘플링 (첫, 중간, 마지막)
    picks_idx = [0, len(visits)//4, len(visits)//2, 3*len(visits)//4, len(visits)-1]
    picks_idx = sorted(set(picks_idx))
    n = len(picks_idx)

    fig, axes = plt.subplots(1, n, figsize=(3.5*n, 3.8))
    if n == 1: axes = [axes]
    for ax, idx in zip(axes, picks_idx):
        v = visits[idx]
        vm = np.array(Image.open(aligned_root / str(pid) / eye / "vessels" / f"{v['id']}.png")) > 0

        H, W = fixed_vessel.shape
        overlay = np.zeros((H, W, 3), dtype=np.uint8)
        overlay[..., 0] = 255 * fixed_vessel   # R = fixed
        overlay[..., 1] = 255 * vm             # G = this visit

        # dice score in ROI
        f_in = fixed_vessel & roi
        v_in = vm & roi
        inter = int((f_in & v_in).sum())
        union = int(f_in.sum() + v_in.sum())
        dice = 2*inter / union if union else np.nan

        ax.imshow(overlay)
        ax.set_title(f"{pd.Timestamp(v['date']).strftime('%Y-%m-%d')}
"
                     f"vs fixed  Dice={dice:.3f}",
                     fontsize=9,
                     color=("green" if dice > 0.5 else "orange" if dice > 0.3 else "red"))
        ax.axis("off")
    fig.suptitle(f"Patient {pid}/{eye} — aligned vessel overlay (R=fixed, G=this visit, Y=overlap)",
                 fontsize=12)
    plt.tight_layout(); plt.show()


**Dice 해석 가이드**:
- **> 0.7** — alignment 우수 (vessel mask 가 대체로 겹침)
- **0.5–0.7** — 정상 (segmentation 노이즈 + 시간 변화 감안)
- **0.3–0.5** — 주의 (alignment 잔차 크거나 quality 저하)
- **< 0.3** — 실패 (registration keypoint 부족했을 가능성. `reg_log.n_kp` 확인)

낮은 Dice 는 **alignment 자체 문제라기보다** 얇은 vessel 모델 예측이 매번 달라져서 그런 경우가 많음.
대혈관 (arcade) 은 overlap 잘 됨.


In [ ]:
# 진단 2: aligned disc mask 의 centroid 가 얼마나 일관되나
from scipy import ndimage as ndi

qc_rows = []
for (pid, eye), roi in patient_roi.items():
    visits = inventory[pid][eye]
    if len(visits) < 2: continue
    fixed = visits[0]
    # fixed disc centroid (aligned = identity)
    fdisc = np.array(Image.open(aligned_root / str(pid) / eye / "discs" / f"{fixed['id']}.png"))
    if fdisc.sum() == 0: continue
    fcy, fcx = ndi.center_of_mass(fdisc > 0)

    for v in visits:
        vd = np.array(Image.open(aligned_root / str(pid) / eye / "discs" / f"{v['id']}.png"))
        if vd.sum() == 0:
            qc_rows.append({"patient":pid,"eye":eye,"id":v["id"],"date":v["date"],
                            "disc_dx":np.nan,"disc_dy":np.nan,"disc_shift_px":np.nan})
            continue
        cy, cx = ndi.center_of_mass(vd > 0)
        dx = cx - fcx; dy = cy - fcy
        qc_rows.append({"patient":pid,"eye":eye,"id":v["id"],"date":v["date"],
                        "disc_dx":dx, "disc_dy":dy,
                        "disc_shift_px":float(np.hypot(dx,dy))})

qc_df = pd.DataFrame(qc_rows)
print("Aligned disc centroid shift (vs fixed 시점, px):")
print(qc_df.groupby(["patient","eye"])["disc_shift_px"].describe().round(1))

# 시각화
fig, ax = plt.subplots(figsize=(12, 4))
for (pid, eye), sub in qc_df.groupby(["patient","eye"]):
    ax.plot(sub["date"], sub["disc_shift_px"], "-o", markersize=4,
            label=f"{pid}/{eye}")
ax.axhline(5,  color="green",  ls="--", alpha=0.5, label="≤5 px (excellent)")
ax.axhline(15, color="orange", ls="--", alpha=0.5, label="≤15 px (ok)")
ax.set_ylabel("disc centroid shift (px)")
ax.set_title("Aligned disc centroid stability (vs fixed visit)")
ax.grid(alpha=0.3); ax.legend(ncol=2, fontsize=8)
ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout(); plt.show()


## 10. 표준 ROI + Vessel feature 추출

**규격화 정책**:
- **`roi_fixed`** : 이미지 중심 반경 `R=230` 원 (10% 만 잘라냄) → **환자 간 규격 통일**
- **`common_valid`** : 이 환자 visit 들의 fundus 픽셀 교집합 → **환자 내 visit 공정 비교**
- **`feature_roi = roi_fixed & common_valid`**

Feature 는 모두 `feature_roi.sum()` 으로 정규화 → alignment 손실 무관.

In [ ]:
# ── 표준 원형 ROI (모든 환자 동일) ──
R_ROI = 230                                              # 반경, 512 grid
cx_img, cy_img = SQUARE_SIZE // 2, SQUARE_SIZE // 2
yy, xx = np.mgrid[:SQUARE_SIZE, :SQUARE_SIZE]
roi_fixed = ((xx - cx_img) ** 2 + (yy - cy_img) ** 2) < R_ROI ** 2

print(f"roi_fixed: R={R_ROI}, area={roi_fixed.sum():,} px "
      f"({100*roi_fixed.sum()/SQUARE_SIZE**2:.1f}% of frame)")

# ── (환자, 좌우안) 별 common_valid 계산 ──
def fundus_valid(rgb_uint8):
    """검은 배경 (fundus 밖) 제외."""
    return rgb_uint8.sum(-1) > 10

def compute_common_valid(visits):
    """이 (환자, 좌우안) 의 모든 visit 에서 fundus 픽셀인 지점만 남김."""
    common = np.ones((SQUARE_SIZE, SQUARE_SIZE), dtype=bool)
    fixed_id = visits[0]["id"]
    for i, v in enumerate(visits):
        rgb = np.array(Image.open(rgb_path / f"{v['id']}.png"))
        if i == 0:
            valid = fundus_valid(rgb)
        else:
            theta = get_theta(v["id"])
            if theta is None: continue
            warped = warp_image(rgb, theta, SQUARE_SIZE, SQUARE_SIZE)
            valid = fundus_valid(warped)
        common &= valid
    return common

# 환자-눈별로 미리 계산 (cache)
patient_roi = {}   # (patient, eye) -> feature_roi (bool 2D)
for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye == "unknown" or len(visits) < 2:
            continue
        common_valid = compute_common_valid(visits)
        feature_roi = roi_fixed & common_valid
        patient_roi[(pid, eye)] = feature_roi
        print(f"  [{pid}/{eye}] common_valid={common_valid.sum():>7,}, "
              f"feature_roi={feature_roi.sum():>7,} "
              f"({100*feature_roi.sum()/roi_fixed.sum():.1f}% of roi_fixed)")

### ROI 시각화 — fixed 이미지 + 원형 boundary + common_valid

In [ ]:
# ROI overlay 시각화 — 환자-눈별 fixed 이미지 위에 원형 boundary + 잘려나가는 영역
n_groups = len(patient_roi)
fig, axes = plt.subplots(1, n_groups, figsize=(5*n_groups, 5))
if n_groups == 1: axes = [axes]

for ax, ((pid, eye), feature_roi) in zip(axes, patient_roi.items()):
    fixed_v = inventory[pid][eye][0]
    fixed_rgb = np.array(Image.open(rgb_path / f"{fixed_v['id']}.png"))

    # 잘려나가는 영역을 빨간색 반투명으로 표시
    excluded = ~feature_roi
    disp = fixed_rgb.copy()
    disp[excluded] = (disp[excluded] * 0.3 + np.array([255, 0, 0]) * 0.7).astype(np.uint8)

    ax.imshow(disp)
    # 원형 boundary
    theta_lin = np.linspace(0, 2*np.pi, 200)
    ax.plot(cx_img + R_ROI*np.cos(theta_lin), cy_img + R_ROI*np.sin(theta_lin),
            color="cyan", linewidth=2, label=f"R={R_ROI}")
    ax.set_title(f"{pid}/{eye} — feature_roi ({feature_roi.sum():,} px)")
    ax.axis("off"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

### 모든 visit 에 ROI 적용 (환자-눈별 timeline)

각 visit 의 aligned 이미지에 동일한 `feature_roi` 를 적용한 결과. ROI 밖은 빨간 반투명, 안은 원본 유지. 이게 실제 feature 가 계산되는 픽셀 영역.

In [ ]:
def apply_roi_overlay(img, roi):
    """ROI 밖 영역을 빨간색 반투명으로."""
    disp = img.copy()
    excluded = ~roi
    disp[excluded] = (disp[excluded] * 0.3 + np.array([255, 0, 0]) * 0.7).astype(np.uint8)
    return disp

def plot_roi_timeline(patient, eye, visits, roi):
    if len(visits) < 2: return
    fixed = visits[0]
    fixed_rgb = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    H, W = fixed_rgb.shape[:2]
    n = len(visits)

    fig, axes = plt.subplots(1, n, figsize=(2.4*n, 2.6))
    if n == 1: axes = [axes]
    theta_lin = np.linspace(0, 2*np.pi, 200)

    for i, v in enumerate(visits):
        if i == 0:
            aligned = fixed_rgb
            title = f"{v['date'].strftime('%Y-%m-%d')}\nFIXED"
        else:
            theta = get_theta(v["id"])
            if theta is None:
                axes[i].axis("off"); continue
            orig = np.array(Image.open(rgb_path / f"{v['id']}.png"))
            aligned = warp_image(orig, theta, H, W)
            title = v['date'].strftime('%Y-%m-%d')

        axes[i].imshow(apply_roi_overlay(aligned, roi))
        axes[i].plot(cx_img + R_ROI*np.cos(theta_lin),
                     cy_img + R_ROI*np.sin(theta_lin),
                     color="cyan", linewidth=0.8)
        axes[i].set_title(title, fontsize=8)
        axes[i].axis("off")

    fig.suptitle(f"Patient {patient} — {eye} eye — feature_roi 적용 ({n} visits, {int(roi.sum()):,} px)",
                 fontsize=11)
    plt.tight_layout(); plt.show()

for (pid, eye), roi in patient_roi.items():
    plot_roi_timeline(pid, eye, inventory[pid][eye], roi)

### Vessel feature 계산 + 시간별 저장

In [ ]:
def vessel_features(vessel_mask, av_mask, disc_mask, roi):
    """ROI 안에서 vessel / A / V / disc density + ratio 계산."""
    roi_n = max(int(roi.sum()), 1)
    v_in  = (vessel_mask > 0) & roi
    art   = np.isin(av_mask, (1, 3)) & roi
    vei   = np.isin(av_mask, (2, 3)) & roi
    disc_in = (disc_mask > 0) & roi
    return {
        "roi_area_px":       roi_n,
        "vessel_area_px":    int(v_in.sum()),
        "vessel_density":    v_in.sum() / roi_n,
        "artery_density":    art.sum()  / roi_n,
        "vein_density":      vei.sum()  / roi_n,
        "AV_ratio_area":     art.sum() / max(int(vei.sum()), 1),
        "disc_area_px":      int(disc_in.sum()),
        "disc_frac":         disc_in.sum() / roi_n,
    }

# 모든 (환자, 눈, visit) 에 대해 aligned mask 로부터 feature 계산
feat_rows = []
for (pid, eye), roi in patient_roi.items():
    aligned_dir = aligned_root / str(pid) / eye
    for v in inventory[pid][eye]:
        vid = v["id"]
        try:
            vs_m = np.array(Image.open(aligned_dir / "vessels" / f"{vid}.png"))
            av_m = np.array(Image.open(aligned_dir / "av"      / f"{vid}.png"))
            dc_m = np.array(Image.open(aligned_dir / "discs"   / f"{vid}.png"))
        except FileNotFoundError:
            continue
        feats = vessel_features(vs_m, av_m, dc_m, roi)
        feat_rows.append({
            "id": vid, "patient": pid, "eye": eye,
            "date": v["date"], "shot": v["shot"],
            "is_fixed": vid == inventory[pid][eye][0]["id"],
            **feats,
        })

features_df = pd.DataFrame(feat_rows).sort_values(["patient", "eye", "date"]).reset_index(drop=True)
features_csv = aligned_root / "features.csv"
features_df.to_csv(features_csv, index=False)
print(f"features 저장: {features_csv} ({len(features_df)} rows)")
features_df.head(10)

### Vessel mask timeline — spike 원인 QC

Feature timeline 에서 튀는 구간이 있으면 실제 mask 를 보면 원인이 보임:
- **낮은 quality 이미지** → 얇은 vessel 을 모델이 잃어버려서 density 급락
- **Registration 실패** → warp 후 ROI 안이 fundus 밖 영역과 겹침
- **Disc segmentation 오작동** → disc 로 잘못 분류된 큰 blob 이 vessel 을 가림

각 visit 마다 `vessel_density` 를 제목에 표시. 값이 크게 튀는 visit 을 시각적으로 특정 가능.

In [ ]:
def plot_vessel_mask_timeline(patient, eye, visits, roi):
    """aligned vessel mask 를 aligned RGB 위에 오버레이 + 제목에 vessel_density."""
    if len(visits) < 2: return
    n = len(visits)
    mdir = aligned_root / str(patient) / eye / "vessels"

    fixed = visits[0]
    fixed_rgb = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    H, W = fixed_rgb.shape[:2]

    sub = features_df[(features_df["patient"] == patient) & (features_df["eye"] == eye)]
    dens_map = dict(zip(sub["id"], sub["vessel_density"]))
    d_median = sub["vessel_density"].median()

    theta_lin = np.linspace(0, 2*np.pi, 200)
    fig, axes = plt.subplots(1, n, figsize=(2.4*n, 2.6))
    if n == 1: axes = [axes]

    for i, v in enumerate(visits):
        vid = v["id"]
        p = mdir / f"{vid}.png"
        if not p.exists():
            axes[i].axis("off"); continue
        mask = np.array(Image.open(p))

        # aligned RGB (warp on-demand)
        if i == 0:
            aligned_rgb = fixed_rgb
        else:
            theta = get_theta(vid)
            if theta is None:
                aligned_rgb = np.array(Image.open(rgb_path / f"{vid}.png"))
            else:
                orig = np.array(Image.open(rgb_path / f"{vid}.png"))
                aligned_rgb = warp_image(orig, theta, H, W)

        # 오버레이: aligned RGB 어둡게 + ROI 밖 붉게 + vessel 초록
        disp = (aligned_rgb.astype(np.float32) * 0.5).astype(np.uint8)
        outside = ~roi
        disp[outside] = (aligned_rgb[outside].astype(np.float32) * 0.15 +
                         np.array([200, 0, 0]) * 0.35).astype(np.uint8)
        disp[(mask > 0) & roi] = [0, 255, 0]

        axes[i].imshow(disp)
        axes[i].plot(cx_img + R_ROI*np.cos(theta_lin),
                     cy_img + R_ROI*np.sin(theta_lin),
                     color="cyan", linewidth=0.6)

        d = dens_map.get(vid, np.nan)
        rel = (d - d_median) / d_median if d_median else 0
        color = "red" if abs(rel) > 0.15 else ("orange" if abs(rel) > 0.08 else "black")
        title = f"{v['date'].strftime('%Y-%m-%d')}vd={d:.3f}"
        axes[i].set_title(title, fontsize=7, color=color)
        axes[i].axis("off")

    fig.suptitle(
        f"Patient {patient} — {eye} eye — aligned vessel (RGB dim + vessel green)  "
        f"median vd = {d_median:.3f}  [빨강 = ±15% 이상 차이]",
        fontsize=11,
    )
    plt.tight_layout(); plt.show()

for (pid, eye), roi in patient_roi.items():
    plot_vessel_mask_timeline(pid, eye, inventory[pid][eye], roi)


### Feature timeline — 환자별 시간에 따른 변화

In [ ]:
# 환자별 vessel/artery/vein density + AV ratio + disc frac timeline
metrics = ["vessel_density", "artery_density", "vein_density", "AV_ratio_area", "disc_frac"]
n_pat = len(patient_ids)

fig, axes = plt.subplots(n_pat, len(metrics), figsize=(4*len(metrics), 3.2*n_pat), squeeze=False)
for row, pid in enumerate(patient_ids):
    for col, metric in enumerate(metrics):
        ax = axes[row, col]
        for eye, style in [("L", "-o"), ("R", "-s")]:
            sub = features_df[(features_df["patient"] == pid) & (features_df["eye"] == eye)]
            if len(sub) == 0: continue
            ax.plot(sub["date"], sub[metric], style, label=eye, markersize=4, linewidth=1)
        ax.set_title(f"{pid} — {metric}")
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout(); plt.show()

## 11. 요약

**저장물**:
- `aligned_root/<patient>/<eye>/{av,vessels,discs}/*.png` — 정렬된 mask (PNG optimize)
- `aligned_root/registration_log.csv` — visit 별 `theta` (t00..t22, 3×3 flat), n_kp
- `aligned_root/features.csv` — visit 별 vessel/artery/vein density, AV ratio, disc frac (표준 ROI 안)

**메모리 상**:
- `inv_df` (DataFrame), `inventory` (dict) — laterality 포함 인벤토리
- `patient_roi` — (환자,눈) → `feature_roi` bool 2D
- `features_df` — feature timeline

**다음 단계**:
- Disc-diameter (DD) 로 length/area 를 정규화해 cross-patient 비교 가능하게
- 시간에 따른 change map (`fixed_mask XOR aligned_mask_t`)
- 더 정교한 vessel feature: skeleton, caliber, fractal dim, tortuosity (`fundus-vessel-features/retinal_graph/*` 이식)
- 더 많은 환자로 확장 (`TOP_N` 늘리기)